# HW1 · Part 3. 상자 DLT → P → K · R · C (15점 · 예상 3–4시간)

**배포본:** v5 (2026-10-20)

**목표** — 치수를 아는 상자 하나로 P 를 DLT 로 구하고, P 를 K · R · C 로 분해해 Part 2 의 K 와 줄자 거리에 맞는지 확인. 평면 점만으로는 왜 안 되는지(퇴화) 직접 체험

**참고** — MVG 2판 §7.1 (DLT 식 (7.1)–(7.2) · 정규화 · 퇴화) · Algorithm 7.1, §6.2.1 · §6.2.4 (카메라 중심 PC = 0 · P = K[R|t] 분해 · Example 6.2), A4.1.1 Algorithm A4.1 (Givens RQ), A5.3 (‖Ap‖ 최소화)

**TODO** — `dlt_matrix` · `dlt_camera` · `camera_center` · `rq_givens` (4개)

**OpenCV 허용** — `cv2.undistortPoints`(클릭 점의 왜곡 제거). `cv2.solvePnP` · `cv2.decomposeProjectionMatrix` 는 **금지**(검증용 비교에만 가능)

## 3.1 상자 좌표계와 코너 클릭
- 상자 치수 (W, D, H) mm 를 실측 → 코너 이름 `'xyz'` 비트: `000` 원점, `100` = x 방향(W) 끝, `010` = y 방향(D), `001` = z 방향(H). 원점은 바닥 코너 하나(보이는 쪽 권장), x · y 는 원점에서 뻗는 두 바닥 모서리, z 는 위 — x × y = z 가 되도록(오른손). 위에서 비스듬히 찍으면 윗면 4개 + 바닥 3개가 보이고 바닥 뒤쪽 코너 하나가 가려짐 (데모 `box1.jpg` 는 `000` 이 가려진 경우)
- 클릭은 아래 셀에서 노트북 안에서 합니다 (ipympl 필요: `pip install ipympl`). 보이지 않는 코너는 이름 목록에서 빼고 6개 이상

> **허용 · 금지** — NumPy · SciPy · Matplotlib 자유. OpenCV 는 각 Part 안내에 "허용" 으로 적힌 함수만(코너 검출 · `findHomography` · 검증용 비교). 
> `TODO` 로 표시된 함수를 OpenCV 호출로 대체하면 해당 항목 0점. 

In [ ]:
# ---- 공통 설정 (HW1 v5 · 모든 노트북에서 같은 설정 파일 사용) ----
import os, sys, json, glob, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if not (ROOT / 'hw1_config.json').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'hw1_config.json').is_file():
    raise FileNotFoundError('HW1 폴더 또는 notebooks 폴더에서 실행하세요: hw1_config.json 을 찾을 수 없습니다.')
sys.path.insert(0, str(ROOT))
from hw1lib import geometry as G, synth as S, exif as E, board as Bd, io_utils as IO, checks as CK, click_tool as CT
# 데모 전환은 HW1/hw1_config.json 의 mode 만 "demo" 로 변경하고 각 노트북의 이 셀을 다시 실행.
cfg = IO.load_config(ROOT)
DATA, RESULTS = cfg['DATA'], cfg['RESULTS']
IO.RESULTS = RESULTS
def load_corners_checked(name):
    """현재 설정과 저장 코너 파일의 검출 배율이 같은지 확인."""
    det = Bd.load_corners(IO.rpath(name))
    if not np.isclose(det['scale'], cfg['SCALE'], rtol=0, atol=1e-12):
        raise ValueError(f"{name}: 저장 배율 {det['scale']}과 현재 설정 scale={cfg['SCALE']}이 다릅니다. Part 0부터 다시 실행하세요.")
    return det
os.makedirs(RESULTS, exist_ok=True)
# 서로 다른 데이터의 결과를 같은 폴더에 섞지 않음. 상대 경로로 기록하여 제출 폴더 이동은 허용.
context = dict(mode=cfg['mode'], data=os.path.relpath(DATA, ROOT), version=5)
context_path = IO.rpath('dataset_context.json')
if os.path.exists(context_path):
    if IO.load_json(context_path) != context:
        raise ValueError('결과 폴더의 데이터 출처가 현재 설정과 다릅니다. 새 results 경로를 지정하고 Part 0부터 실행하세요.')
elif any(not p.name.startswith('.') for p in Path(RESULTS).iterdir()):
    raise ValueError('출처 기록이 없는 기존 결과가 있습니다. 새 results 경로를 지정하고 Part 0부터 실행하세요.')
IO.save_json(context_path, context)
np.set_printoptions(precision=4, suppress=True)
import matplotlib
_fonts = {f.name for f in matplotlib.font_manager.fontManager.ttflist}
for _f in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR', 'Freesentation']:
    if _f in _fonts:
        matplotlib.rcParams['font.family'] = _f; break
matplotlib.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore', message='Glyph')
%matplotlib inline
print('mode =', cfg['mode'], '\nDATA =', DATA, '\nRESULTS =', RESULTS)


### 클릭 — 처음 한 번만 (JSON 이 있으면 건너뜀)
왼쪽 클릭 = 점, 오른쪽 클릭 = 취소. 툴바 돋보기로 확대한 뒤(돋보기 해제 후) 클릭하면 서브픽셀 정밀도. 클릭을 마치면 **다음 셀**을 실행해 저장.

In [ ]:
box_json = os.path.join(DATA, 'box', 'box1_points.json')
BOX_NAMES = ['000', '100', '010', '001', '101', '111', '011']       # 보이는 코너 7개 (윗면 4 + 바닥 3, 가려진 '110' 제외) — 본인 사진에 맞게 수정
if not os.path.exists(box_json):
    get_ipython().run_line_magic('matplotlib', 'widget')
    clicker = CT.NotebookClicker(os.path.join(DATA, 'box', 'box1.jpg'), kind='points', names=BOX_NAMES, title='상자 코너').show()
else:
    print('이미 있음:', box_json)

In [ ]:
if not os.path.exists(box_json):
    clicker.save(box_json)
    get_ipython().run_line_magic('matplotlib', 'inline')
box2_json = os.path.join(DATA, 'box', 'box2_oneface_points.json')          # 윗면만 보이는 사진: 윗면 코너 4개 (다른 면을 찍었으면 이름을 그 면의 코너로)
if not os.path.exists(box2_json):
    get_ipython().run_line_magic('matplotlib', 'widget')
    clicker2 = CT.NotebookClicker(os.path.join(DATA, 'box', 'box2_oneface.jpg'), kind='points', names=['001', '101', '111', '011'], title='한 면 코너').show()
else:
    print('이미 있음:', box2_json)

In [ ]:
if not os.path.exists(box2_json):
    clicker2.save(box2_json)
    get_ipython().run_line_magic('matplotlib', 'inline')

In [ ]:
# Part 2의 K를 대상 이미지의 픽셀 격자로 변환한 뒤 클릭점을 역왜곡.
import cv2
K_cal, kinfo = IO.load_K('K.json'); dist_z, _ = IO.load_dist('dist.json')
def image_size_of(path):
    h, w = E.load_image(path, gray=True).shape[:2]
    return (w, h)
def K_for_image(path):
    return IO.rescale_K_for_image(K_cal, kinfo['image_size'], image_size_of(path))
def undistort_uv(uv, K_image):
    uv = np.asarray(uv, float).reshape(-1, 1, 2)
    return cv2.undistortPoints(uv, K_image, dist_z, P=K_image).reshape(-1, 2)
print('K · dist 로드:', kinfo['source'], '캘리브레이션 크기:', kinfo['image_size'])


In [ ]:
meas = IO.load_json(IO.rpath('measurements.json'))
dims = np.asarray(meas['box_dims_mm'], float)
if dims.shape != (3,) or not np.all(np.isfinite(dims) & (dims > 0)):
    raise ValueError('measurements.json의 box_dims_mm에 양수 세 변 길이(mm)를 입력하세요.')
rec = IO.load_json(os.path.join(DATA, 'box', 'box1_points.json'))
names = rec['names']; uv_raw = np.array(rec['points'], float)
X8 = S.box_corners(dims); idx = [S.BOX_CORNER_NAMES.index(n) for n in names]
Xw = X8[idx]
box_image = os.path.join(DATA, 'box', rec['image'])
box_image_size = image_size_of(box_image)
K_z = K_for_image(box_image)
uv = undistort_uv(uv_raw, K_z)
print('상자 치수', dims, 'mm, 코너', len(names), '개:', names)
CT.show_points(os.path.join(DATA, 'box', rec['image']), uv_raw, names);

## 3.2 정규화 DLT — A 행렬 · SVD · 되돌리기 (5점)
식 (7.2) 의 2n × 12 행렬을 쌓고, 정규화(MVG §7.1 "Data normalization")한 뒤 SVD 의 최소 특이벡터(A5.3)로 P 를 구합니다. 정규화 유무의 **조건수** 를 비교하세요.

In [ ]:
# ==============================================================================
# TODO  dlt_matrix
# 참조: MVG §7.1 식 (7.1)–(7.2) (외적으로 스케일을 없애면 P 에 대한 1차식 — 블록을 쌓으면 2n × 12)
# 점마다 두 행 (x = (x, y, w=1), X = (X, Y, Z, 1)):  [ 0ᵀ,  −w Xᵀ,   y Xᵀ ]  과  [ w Xᵀ,  0ᵀ,  −x Xᵀ ]
# p = (P 의 1행, 2행, 3행) 을 이은 12-벡터.  반환 A (2n × 12)
# ==============================================================================
def dlt_matrix(Xw, uv):
    """식 (7.2) 의 2n × 12 행렬 A.  각 점: [0ᵀ, −w Xᵀ, y Xᵀ ; w Xᵀ, 0ᵀ, −x Xᵀ]  (MVG §7.1)"""
    # ---- 여기에 작성 ----
    raise NotImplementedError('TODO: dlt_matrix')


# ==============================================================================
# TODO  dlt_camera
# 참조: MVG Algorithm 7.1 (정규화 · A · SVD · 되돌리기), §7.1 "Data normalization", A5.3 SVD 풀이
# ① 정규화: 2D 점에 T, 3D 점에 U (geometry.normalization_transform — 중심 원점 · 평균 거리 √2 / √3)   ② A 구성   ③ SVD → 최소 특이값의 우특이벡터 → P̃ (3×4)
# ④ 되돌리기 P = T⁻¹ P̃ U   ⑤ 부호: 점이 카메라 앞(깊이 > 0)에 오도록 — P[2,:] · X̃ 가 대부분 음수면 P ← −P   ⑥ ‖P[2, :3]‖ = 1 로 스케일
# normalize=False 일 때와 특이값(조건수 s[0]/s[-2])을 비교해 보고서에 — 정규화 없이는 조건수가 10⁵ 이상
# 반환 (P, s) — s의 길이는 min(2n, 12). 4점 퇴화 실험에서는 8개.
# 4점의 8×12 행렬에서도 우영공간을 얻으려면 np.linalg.svd(A, full_matrices=True)를 사용.
# 정상 추정에는 비평면 점 6개 이상이 필요하지만, 3.5는 퇴화 진단을 위해 4점에도 실행.
# ==============================================================================
def dlt_camera(Xw, uv, normalize=True):
    """정규화 DLT (MVG Algorithm 7.1): T (2D) · U (3D) 정규화 → A → SVD → P̃ = 최소 특이벡터 → P = T⁻¹ P̃ U.
    반환 P (3×4, 관측점의 깊이 부호 정리), 특이값 s (min(2n, 12),)"""
    # ---- 여기에 작성 ----
    raise NotImplementedError('TODO: dlt_camera')


In [ ]:
# 제공 — 재투영 오차 (px)
def reprojection_errors(P, Xw, uv):
    x = P @ G.to_homog(np.asarray(Xw, float).T)
    return np.linalg.norm((x[:2] / x[2]).T - uv, axis=1)


In [ ]:
P, sv = dlt_camera(Xw, uv, normalize=True)
P_un, sv_un = dlt_camera(Xw, uv, normalize=False)
err = reprojection_errors(P, Xw, uv); err_un = reprojection_errors(P_un, Xw, uv)
print('정규화 O: 특이값', sv.round(4), '\n  조건수 s1/s11 = %.1f, 재투영 RMS %.3f px (최대 %.3f)' % (sv[0] / sv[-2], np.sqrt(np.mean(err ** 2)), err.max()))
print('정규화 X: 특이값', sv_un.round(2), '\n  조건수 s1/s11 = %.1e, 재투영 RMS %.3f px' % (sv_un[0] / sv_un[-2], np.sqrt(np.mean(err_un ** 2))))
print('P =\n', P)
CK.check_dlt(dlt_camera);

## 3.3 분해 — 카메라 중심 · RQ → K, R (5점)
C̃ = −M⁻¹p₄ (MVG §6.2.1), M = KR 은 Givens 회전 세 번의 RQ (Algorithm A4.1). `decompose_P` 는 제공(두 TODO 를 조립).

In [ ]:
# ==============================================================================
# TODO  camera_center
# 참조: MVG §6.2.1 (카메라 중심 PC = 0 — 유한 카메라 C̃ = −M⁻¹p₄), §6.2.4
# 유한 카메라: P = [M | p4] → C̃ = −M⁻¹ p4.  (또는 SVD 의 영공간, 동차 4-벡터를 정규화)
# ==============================================================================
def camera_center(P):
    """PC = 0 — SVD 의 영공간, 또는 유한 카메라면 C̃ = −M⁻¹ p4  (MVG §6.2.1)"""
    # ---- 여기에 작성 ----
    raise NotImplementedError('TODO: camera_center')


# ==============================================================================
# TODO  rq_givens
# 참조: MVG A4.1.1 Algorithm A4.1 (Givens 회전 세 번으로 RQ), §6.2.4
# M (3×3) 에 오른쪽에서 Qx, Qy, Qz 를 차례로 곱해 M32, M31, M21 을 0 으로:  M Qx Qy Qz = K (상삼각) → M = K R,  R = (Qx Qy Qz)ᵀ
# Qx = [[1,0,0],[0,c,−s],[0,s,c]]:  (M Qx)[2,1] = M21·c + M22·s = 0  →  c = M22/√(M21²+M22²), s = −M21/√(…)  (인덱스는 0 기반: 행 2, 열 1·2)
# Qy = [[c,0,s],[0,1,0],[−s,0,c]]:  (M Qy)[2,0] = M20·c − M22·s = 0  →  c = M22/d, s = M20/d
# Qz = [[c,−s,0],[s,c,0],[0,0,1]]:  (M Qz)[1,0] = M10·c + M11·s = 0  →  c = M11/d, s = −M10/d
# 부호 정리: K 의 대각이 음수인 열은 K 의 그 열과 R 의 그 행에 −1 → K 대각 양.  det R = −1 이면 R ← −R.  K 는 K[2,2] = 1 로 정규화
# 대안: scipy.linalg.rq 사용 가능 — 단 부호 정리(대각 양 · det +1)를 직접 하지 않으면 감점.  반환 (K, R)
# ==============================================================================
def rq_givens(M):
    """Algorithm A4.1 (MVG A4.1.1): Givens 회전 Qx, Qy, Qz 로 M 의 아래 삼각 성분을 차례로 0 으로 → M = K R.
    K 의 대각이 양이 되도록 부호 정리, det R = +1."""
    # ---- 여기에 작성 ----
    raise NotImplementedError('TODO: rq_givens')


In [ ]:
# 제공 — C̃ · RQ 조립, t = −R C̃
def decompose_P(P):
    """P = K[R | t] 분해 (MVG §6.2.4): C̃, M = KR 을 RQ 로, t = −R C̃."""
    C = camera_center(P)
    K, R = rq_givens(P[:, :3])
    if np.linalg.det(R) < 0:
        R = -R
    t = -R @ C
    return K, R, t, C


In [ ]:
K_dlt, R_dlt, t_dlt, C_dlt = decompose_P(P)
print('K_dlt =\n', K_dlt, '\nR_dlt =\n', R_dlt, '\nC (상자 좌표, mm) =', C_dlt)
import scipy.linalg
Ks, Rs_ = scipy.linalg.rq(P[:, :3]); print('scipy.linalg.rq 의 대각 부호 (정리 전):', np.sign(np.diag(Ks)))
CK.check_rq(rq_givens); CK.check_dlt(dlt_camera, decompose_P);

## 3.4 실측과 Part 2 와 비교 (3점)

In [ ]:
d_est = np.linalg.norm(C_dlt - X8.mean(axis=0)); d_meas = meas.get('box_distance_mm')
print('카메라 ~ 상자 중심 거리: 추정 %.1f mm' % d_est + (', 실측 %.1f mm, 오차 %+.1f %%' % (d_meas, (d_est - d_meas) / d_meas * 100) if d_meas else ''))
if K_z is not None:
    print('f: DLT %.1f / %.1f  vs  Zhang(C) %.1f / %.1f  → 차이 %+.1f %% / %+.1f %%' % (K_dlt[0, 0], K_dlt[1, 1], K_z[0, 0], K_z[1, 1], (K_dlt[0, 0] / K_z[0, 0] - 1) * 100, (K_dlt[1, 1] / K_z[1, 1] - 1) * 100))
    print('c: DLT (%.1f, %.1f)  vs  Zhang (%.1f, %.1f),  skew_dlt = %.2f' % (K_dlt[0, 2], K_dlt[1, 2], K_z[0, 2], K_z[1, 2], K_dlt[0, 1]))
IO.save_json(IO.rpath('dlt.json'), dict(image_size=list(box_image_size), image=rec['image'], P=P, K=K_dlt, R=R_dlt, t=t_dlt, C=C_dlt, rms=float(np.sqrt(np.mean(err ** 2))), singular_values=sv,
                                        cond_normalized=sv[0] / sv[-2], cond_unnormalized=sv_un[0] / sv_un[-2], distance_est=d_est, distance_measured=d_meas, n_points=len(names)))

**보고서 질문 3.4** — DLT 의 f 가 Zhang 보다 (보통 수 %) 다릅니다. 점 7개 · 작은 깊이 범위 · 클릭 오차 ~1 px 가 f · C 에 어떻게 전파되는지(DLT 의 한계, 제약 카메라 — MVG §7.3),
그리고 "재투영 오차는 작은데 K 는 틀린" 상황이 왜 생기는지(재투영 오차가 말하지 못하는 것) 한 문단.

## 3.5 퇴화 체험 — 한 면만 · 평면 위의 점 (2점)
(a) 한 면의 코너 4개: 식 8개 · 미지수 12 → 영공간이 4차원. (b) 보드 코너 54개(모두 Z = 0): 점이 많아도 **한 평면** 이면 P 는 결정되지 않음(MVG §7.1 퇴화 배치) — 특이값을 보세요.

In [ ]:
rec2 = IO.load_json(os.path.join(DATA, 'box', 'box2_oneface_points.json'))
Xw2 = X8[[S.BOX_CORNER_NAMES.index(n) for n in rec2['names']]]
K_box2 = K_for_image(os.path.join(DATA, 'box', rec2['image']))
uv2 = undistort_uv(np.array(rec2['points'], float), K_box2)
def degeneracy_report(label, Xw_, uv_):
    A_ = dlt_matrix(*[np.asarray(a, float) for a in (Xw_, uv_)])
    sv_ = np.linalg.svd(A_, compute_uv=False)
    rank = np.linalg.matrix_rank(A_, tol=1e-8 * sv_[0])
    print(f'{label}: A 는 {A_.shape[0]} × 12, rank {rank} → 영공간 {12 - rank} 차원 (P 가 하나로 정해지려면 1 이어야 함)')
    print('   특이값 끝 5개:', np.round(sv_[-5:], 6))
    try:
        P_, _ = dlt_camera(Xw_, uv_)
    except (ValueError, np.linalg.LinAlgError) as e:
        print('   추정기가 퇴화 입력을 거부함:', e); return
    with np.errstate(all='ignore'):
        e_ = reprojection_errors(P_, Xw_, uv_)
    finite = np.isfinite(e_)
    if finite.any():
        print('   재투영 RMS %.2e px (유효 %d/%d점; 0이어도 P는 임의의 해 하나)' % (np.sqrt(np.mean(e_[finite] ** 2)), finite.sum(), len(e_)))
    else:
        print('   재투영 불능: 모든 점의 투영 분모가 0 또는 비유한 값. 퇴화 P의 한 해입니다.')
    try:
        print('   분해한 K =\n', decompose_P(P_)[0])
    except Exception as e:
        print('   분해 실패:', e)
degeneracy_report('(a) 한 면 4점', Xw2, uv2)
det = load_corners_checked('corners_setC.npz')
K_board = IO.rescale_K_for_image(K_cal, kinfo['image_size'], det['image_size'])
degeneracy_report('(b) 보드 점 (한 평면)', det['objpoints'][0], undistort_uv(det['imgpoints'][0], K_board))

## 보고서에 넣을 것 (Part 3, 슬라이드 3장)
1. 상자 좌표계 그림(코너 이름) + 클릭 사진 + 정규화 O/X 조건수 · 재투영 표
2. 분해 결과: K_dlt vs K_zhang, 거리 추정 vs 실측, 질문 3.4 답
3. 퇴화: (a) · (b) 특이값과 "왜" (한 문단)

In [ ]:
CK.run_all(globals());